# Демография: бренды

Пример расчета социально-демографической структуры аудитории по отдельным брендам или совокупности брендов внутри категории

## Описание задачи и условий расчета
- Период: весь доступный период
- Категории: все доступные категории (в примере "Смартфоны")
- Бренды: средние и крупные бренды доступных категорий
- Статистики: количество запросов (тыс), социально-демографическая структура аудитории (%)

### Инициализация

Импортируем библиотеки, которые понадобятся для работы.\
Выполните следующую ячейку, для этого перейдите в нее и нажмите Shift + Enter

In [ ]:
import pandas as pd
import numpy as np

from fileapiback import *

### Формирование задания

Зададим условия для расчета\
*см. подробнее формат условий в блокноте [help.ipynb](help.ipynb)*

In [ ]:
# укажите период в формате:
# None (все месяцы с активностью и соц.-дем в архиве) или ['YYYY-MM-01', 'YYYY-MM-01'] (конкретные месяцы)
period = None

# укажите одну категорию брендов для фильтрации в формате: 'Категория'
category_filter = 'Смартфоны'

# укажите бренды из категории для фильтрации в формате:
# None (все бренды) или  ['бренд1', 'бренд2']
brand_filter = None

# укажите, нужна ли разбивка по брендам
# да - True, нет - False
brand_flag = True

# укажите один соц.-дем параметр
# 'Sex' (пол) или 'Age_group' (возраст) или 'Income' (доход) или 'Region' (регион)
demo_filter = 'Sex'

### Выполнение задания
- Расчет
  - количества запросов брендов или их совокупности в разбивке по соц.-дем параметрам по месяцам (тыс)
  - социально-демографической структуры аудитории брендов или их совокупности по месяцам (%)

In [ ]:
# формирование pandas.DataFrame

obj = FileApi()

# создаем pandas.DataFrame
# датасет отфильтрован с учетом минимально допустимого размера выборки для анализа
data_filt = obj.create_pdDF(note='socdem', period=period, category_filter=category_filter,
                                brand_filter=brand_filter, 
                                demo_filter=demo_filter, brand_flag=brand_flag)

In [ ]:
# расчет количества запросов и социально-демографической структуры аудитории

# считаем количество запросов
qcnt = (data_filt
             .groupby(['month', 'Category', 'Brand', demo_filter])['Weight'] # группируем по месяцу, категории, бренду, соц.-дем параметру и считаем количество запросов
                        .sum() # сумма по группе
                        .reset_index(name='QCnt_param')) # переименование столбца

qcnt['QCnt'] = qcnt.groupby(['month', 'Category', 'Brand'])['QCnt_param'].transform('sum') # добавляем общую сумму

# округляем показатели
qcnt[['QCnt000', 'QCnt000_param']] = (qcnt[['QCnt', 'QCnt_param']] / 1000)

# определяем структуру (%)
qcnt['%'] = (qcnt['QCnt_param'] / qcnt['QCnt'] * 100).round(1) # вычисляем процент

# формируем таблицу для выгрузки
qcnt_pivot = obj.create_pivot(qcnt, demo_filter, ['QCnt000_param', '%'])
qcnt_pivot = qcnt_pivot.sort_values(['month', 'Brand', 'QCnt000'], ascending=[True, True, False]) # сортировка

## Экспорт в Excel
По умолчанию файл 'demography-metrics-YYYYMMDD_HHMMSS.xlsx' сохраняется в ту же директорию, где лежат все блокноты

In [ ]:
time_now = datetime.now().strftime('%Y%m%d_%H%M%S')

with pd.ExcelWriter(f'demography-metrics-{time_now}.xlsx') as writer:
    qcnt_pivot.to_excel(writer, 'demography_metrics', index=False)